## Pattern 19 : String Matching (KMP)

### 🔍 What is KMP?

The **Knuth-Morris-Pratt (KMP)** algorithm is an efficient string matching algorithm that searches for occurrences of a **pattern** within a **text**.

**Core Idea:** Instead of re-comparing characters you've already matched (like brute force does), KMP uses information about the pattern itself to *skip ahead* intelligently when a mismatch occurs.

---

### 🧠 Intuition: Why is Brute Force Wasteful?

Consider searching for pattern `ABABC` in text `ABABABABC`:

```
Text:    A B A B A B A B C
Pattern: A B A B C
                 ^ mismatch at index 4
```

Brute force would shift the pattern by **1 position** and restart comparison from scratch. But we already know that:
- The last two matched characters `AB` (at positions 2-3) are the **same** as the beginning of the pattern `AB`.
- So we can *slide* the pattern forward by 2, aligning the prefix `AB` with those already-matched characters — **no need to re-compare them!**

```
Text:    A B A B A B A B C
Pattern:     A B A B C        ← shifted smartly, reuse matched prefix
```

This is exactly what KMP does. It pre-computes a table (called **LPS**) to know how far to jump.

---

### 📐 Step 1: Build the LPS (Longest Prefix Suffix) Array

The **LPS array** is the heart of KMP. For each position `i` in the pattern, `lps[i]` stores the **length of the longest proper prefix** of `pattern[0..i]` that is also a **suffix**.

| Term | Meaning |
|---|---|
| **Prefix** | A substring starting from index 0 (e.g., `A`, `AB`, `ABA` for `ABABC`) |
| **Suffix** | A substring ending at the last index (e.g., `C`, `BC`, `ABC` for `ABABC`) |
| **Proper** | The prefix/suffix cannot be the entire string itself |

**Example — Pattern: `A B A B C`**

| Index | Substring | Longest Proper Prefix = Suffix | LPS Value |
|:---:|:---:|:---:|:---:|
| 0 | `A` | (none) | 0 |
| 1 | `AB` | (none) | 0 |
| 2 | `ABA` | `A` | 1 |
| 3 | `ABAB` | `AB` | 2 |
| 4 | `ABABC` | (none) | 0 |

**LPS = `[0, 0, 1, 2, 0]`**

#### How the LPS is Built (Step by Step)

We use **two pointers**: `i` (current position) and `length` (length of previous longest prefix-suffix).

1. Start with `lps[0] = 0`, `length = 0`, `i = 1`.
2. **If `pattern[i] == pattern[length]`:** we've extended the match → `length += 1`, `lps[i] = length`, `i += 1`.
3. **If mismatch and `length != 0`:** don't give up — fall back to `lps[length - 1]` (try a shorter prefix-suffix).
4. **If mismatch and `length == 0`:** no prefix-suffix exists → `lps[i] = 0`, `i += 1`.

---

### 🔄 Step 2: Search Using the LPS Array

With the LPS array ready, the search proceeds as:

1. Maintain two pointers: `i` (index in text) and `j` (index in pattern).
2. **Characters match:** advance both `i` and `j`.
3. **Full match found (`j == len(pattern)`):** return the starting index `i - j`.
4. **Mismatch:**
   - If `j != 0`: instead of resetting `j` to 0, set `j = lps[j - 1]`. This *reuses* the already matched prefix.
   - If `j == 0`: just advance `i` (no match at all at this position).

#### Walkthrough

Text = `ABABABABC`, Pattern = `ABABC`, LPS = `[0, 0, 1, 2, 0]`

```
Step 1: i=0,j=0 → A==A ✓ → i=1,j=1
Step 2: i=1,j=1 → B==B ✓ → i=2,j=2
Step 3: i=2,j=2 → A==A ✓ → i=3,j=3
Step 4: i=3,j=3 → B==B ✓ → i=4,j=4
Step 5: i=4,j=4 → A≠C ✗ → j=lps[3]=2  (reuse prefix 'AB')
Step 6: i=4,j=2 → A==A ✓ → i=5,j=3
Step 7: i=5,j=3 → B==B ✓ → i=6,j=4
Step 8: i=6,j=4 → A≠C ✗ → j=lps[3]=2  (reuse prefix 'AB')
Step 9: i=6,j=2 → A==A ✓ → i=7,j=3
Step 10: i=7,j=3 → B==B ✓ → i=8,j=4
Step 11: i=8,j=4 → C==C ✓ → j==5 == len(pattern) → Found at index 4!
```

---

### 🎯 Key Takeaway

| Aspect | Brute Force | KMP |
|---|---|---|
| On mismatch | Go back and restart | Jump ahead using LPS |
| Text pointer `i` | Can go backwards | **Never goes backwards** |
| Time Complexity | O(n × m) | O(n + m) |

In [1]:
def kmp_search(text, pattern):
    """KMP pattern matching"""
    def compute_lps(pattern):
        lps = [0] * len(pattern)
        length = 0
        i = 1
        while i < len(pattern):
            if pattern[i] == pattern[length]:
                length += 1
                lps[i] = length
                i += 1
            elif length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1
        return lps
    
    lps = compute_lps(pattern)
    i = j = 0

    while i < len(text):
        if pattern[j] == text[i]:
            i += 1
            j += 1
        if j == len(pattern):
            return i - j
        elif i < len(text) and pattern[j] != text[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    
    return -1

### ▶️ Example Usage

In [2]:
# --- Example 1: Pattern found ---
text1 = "ABABABABC"
pattern1 = "ABABC"
result1 = kmp_search(text1, pattern1)
print(f"Text: '{text1}' | Pattern: '{pattern1}'")
print(f"Found at index: {result1}")       # Expected: 4
print()

# --- Example 2: Pattern at the beginning ---
text2 = "HELLOWORLD"
pattern2 = "HELLO"
result2 = kmp_search(text2, pattern2)
print(f"Text: '{text2}' | Pattern: '{pattern2}'")
print(f"Found at index: {result2}")       # Expected: 0
print()

# --- Example 3: Pattern not found ---
text3 = "ABCDEFGH"
pattern3 = "XYZ"
result3 = kmp_search(text3, pattern3)
print(f"Text: '{text3}' | Pattern: '{pattern3}'")
print(f"Found at index: {result3}")       # Expected: -1
print()

# --- Example 4: Overlapping pattern ---
text4 = "AAAAAB"
pattern4 = "AAAB"
result4 = kmp_search(text4, pattern4)
print(f"Text: '{text4}' | Pattern: '{pattern4}'")
print(f"Found at index: {result4}")       # Expected: 2

Text: 'ABABABABC' | Pattern: 'ABABC'
Found at index: 4

Text: 'HELLOWORLD' | Pattern: 'HELLO'
Found at index: 0

Text: 'ABCDEFGH' | Pattern: 'XYZ'
Found at index: -1

Text: 'AAAAAB' | Pattern: 'AAAB'
Found at index: 2


### ⏱️ Time & Space Complexity

Let **n** = length of text, **m** = length of pattern.

| Phase | Time | Space |
|---|---|---|
| Building LPS array | **O(m)** | **O(m)** (to store the LPS array) |
| Searching text | **O(n)** | **O(1)** (only two pointers) |
| **Total** | **O(n + m)** | **O(m)** |

#### Why O(n + m) and not O(n × m)?

The key insight is that the text pointer `i` **never moves backwards**. In brute force, after a mismatch you reset `i` back and try again — this causes worst-case O(n × m). In KMP:

- `i` only moves **forward** (at most `n` moves total).
- `j` can fall back via LPS, but each fallback corresponds to a previous advance — so `j` moves at most **2m** times across the entire search.
- Building the LPS array uses the same logic, processing each of the `m` characters at most twice → **O(m)**.

**Result:** The total work is bounded by **O(n + m)**, making KMP optimal for single-pattern string matching.

#### When to Use KMP

| Scenario | Use KMP? |
|---|---|
| Single pattern in a large text | ✅ Yes — O(n + m) is much better than O(n × m) |
| Pattern has repeating prefixes (e.g., `ABABAB...`) | ✅ Yes — LPS gives the biggest speedup here |
| Multiple different patterns in one text | ❌ Use **Aho-Corasick** instead |
| Very short text / pattern | ⚠️ Brute force is fine (overhead of LPS not worth it) |